<a href="https://colab.research.google.com/github/emirbekk08/data-analysis-practice/blob/os-practicals-only/project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os

# ==========================================
# ШАГ 1: Создание папки и файлов с процессами
# ==========================================
os.makedirs("project1", exist_ok=True)

# Записываем тестовые данные процессов (PID arrival burst priority)
with open("project1/set1.txt", "w") as f:   f.write("1 0 5 3\n2 2 3 1\n3 4 1 2")
with open("project1/jobs.txt", "w") as f:   f.write("1 0 10 2\n2 1 4 1\n3 2 6 3")
with open("project1/mixed.txt", "w") as f:  f.write("1 0 15 4\n2 3 2 1\n3 5 4 2")
with open("project1/starve.txt", "w") as f: f.write("1 0 20 5\n2 1 2 1\n3 2 2 1")

print("✅ Папка project1 и 4 файла успешно созданы программно!")

# ==========================================
# ШАГ 2: Описание структуры Процесса
# ==========================================
class Process:
    def __init__(self, pid, arrival, burst, priority):
        self.pid = int(pid)
        self.arrival = int(arrival)
        self.burst = int(burst)
        self.priority = int(priority)
        self.remaining_time = int(burst)
        self.waiting_time = 0
        self.turnaround_time = 0
        self.completion_time = 0

# ==========================================
# ШАГ 3: Парсер файлов
# ==========================================
def load_processes(file_path):
    processes = []
    if not os.path.exists(file_path):
        return processes
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or any(word in line for word in ["Строка", "PID", "arrival"]):
                continue
            parts = line.split()
            if len(parts) == 4:
                processes.append(Process(parts[0], parts[1], parts[2], parts[3]))
    return processes

# ==========================================
# ШАГ 4 & 5: Алгоритм и Симуляция (Round Robin + Приоритеты)
# ==========================================
def simulate_scheduler(processes, quantum=2):
    current_time = 0
    ready_queue = []
    unfinished = processes.copy()
    gantt_chart = []

    while unfinished:
        # Добавляем вновь прибывшие процессы
        for p in list(unfinished):
            if p.arrival <= current_time and p not in ready_queue and p.remaining_time > 0:
                ready_queue.append(p)

        # Сортировка по приоритету (меньше число = выше приоритет), затем по времени прибытия
        ready_queue.sort(key=lambda x: (x.priority, x.arrival))

        if not ready_queue:
            gantt_chart.append("Idle")
            current_time += 1
            continue

        current_process = ready_queue.pop(0)
        time_to_run = min(quantum, current_process.remaining_time)

        for _ in range(time_to_run):
            gantt_chart.append(f"P{current_process.pid}")
            current_time += 1

            # Проверяем новые процессы во время выполнения кванта
            for p in list(unfinished):
                if p.arrival <= current_time and p not in ready_queue and p != current_process and p.remaining_time > 0:
                    ready_queue.append(p)
            ready_queue.sort(key=lambda x: (x.priority, x.arrival))

        current_process.remaining_time -= time_to_run

        if current_process.remaining_time == 0:
            current_process.completion_time = current_time
            current_process.turnaround_time = current_process.completion_time - current_process.arrival
            current_process.waiting_time = current_process.turnaround_time - current_process.burst
            unfinished.remove(current_process)
        else:
            ready_queue.append(current_process)

    return gantt_chart

# ==========================================
# ШАГ 6: Вывод готового отчета
# ==========================================
def run_project_analysis(file_name):
    print(f"\n{'='*50}\n📊 АНАЛИЗ ФАЙЛА: {file_name}\n{'='*50}")
    proc_list = load_processes(f"project1/{file_name}")

    if not proc_list:
        print("Ошибка: не удалось загрузить процессы.")
        return

    chart = simulate_scheduler(proc_list, quantum=2)

    print("📈 Диаграмма Ганта (Порядок работы процессора):")
    print(" -> ".join(chart))

    print("\n📋 Таблица результатов:")
    total_w, total_t = 0, 0
    for p in proc_list:
        print(f"  [PID {p.pid}] Прибыл: {p.arrival} | Время выполнения: {p.burst} | Приоритет: {p.priority} || Время ожидания: {p.waiting_time} | Полное время: {p.turnaround_time}")
        total_w += p.waiting_time
        total_t += p.turnaround_time

    print(f"\n⏱️ Среднее время ожидания (Average Waiting Time): {total_w / len(proc_list):.2f}")
    print(f"⏱️ Среднее полное время (Average Turnaround Time): {total_t / len(proc_list):.2f}")

# Запускаем симуляцию для набора mixed.txt в качестве демонстрации
# Запускаем анализ по очереди для каждого из 4 текстовых файлов проекта
files_to_analyze = ["set1.txt", "jobs.txt", "mixed.txt", "starve.txt"]

for file_name in files_to_analyze:
    run_project_analysis(file_name)




✅ Папка project1 и 4 файла успешно созданы программно!

📊 АНАЛИЗ ФАЙЛА: set1.txt
📈 Диаграмма Ганта (Порядок работы процессора):
P1 -> P1 -> P2 -> P2 -> P2 -> P3 -> P1 -> P1 -> P1

📋 Таблица результатов:
  [PID 1] Прибыл: 0 | Время выполнения: 5 | Приоритет: 3 || Время ожидания: 4 | Полное время: 9
  [PID 2] Прибыл: 2 | Время выполнения: 3 | Приоритет: 1 || Время ожидания: 0 | Полное время: 3
  [PID 3] Прибыл: 4 | Время выполнения: 1 | Приоритет: 2 || Время ожидания: 1 | Полное время: 2

⏱️ Среднее время ожидания (Average Waiting Time): 1.67
⏱️ Среднее полное время (Average Turnaround Time): 4.67

📊 АНАЛИЗ ФАЙЛА: jobs.txt
📈 Диаграмма Ганта (Порядок работы процессора):
P1 -> P1 -> P2 -> P2 -> P2 -> P2 -> P1 -> P1 -> P1 -> P1 -> P1 -> P1 -> P1 -> P1 -> P3 -> P3 -> P3 -> P3 -> P3 -> P3

📋 Таблица результатов:
  [PID 1] Прибыл: 0 | Время выполнения: 10 | Приоритет: 2 || Время ожидания: 4 | Полное время: 14
  [PID 2] Прибыл: 1 | Время выполнения: 4 | Приоритет: 1 || Время ожидания: 1 | Полно